# Assignment 2

This notebook builds upon the data preparation and exploratory analyses performed in Assignment 1. The aim is to engineer machine learning features from the matched genomic and drug response datasets before developing predictive models of Trametinib sensitivity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    GridSearchCV,
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error,
)

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor

# 6. Model Development 

## 6.1. Load Processed Dataset

In [ ]:
# Read the merged_gene_level.csv file exported from notebook1
merged_gene_level = pd.read_csv("../data/merged_gene_level.csv")

print(merged_gene_level.shape)
merged_gene_level.head()

In [ ]:
# Inspect what we have in terms of potential features
print(merged_gene_level.columns.tolist())
print(merged_gene_level.info())

## 6.2. Feature Engineering

The cleaned gene-level dataset produced in Notebook 1 was used as the starting point for machine learning feature engineering.

Before constructing the feature matrix, the dataset was validated to confirm the expected number of unique cancer cell lines and genes, and to ensure that no duplicate cell line–gene combinations remained following preprocessing. These checks verified that each row represented a unique gene mutation within an individual cancer cell line and that the dataset was suitable for conversion from long to wide format for machine learning.

In [ ]:
print(f"Number of unique cell lines: {merged_gene_level['SANGER_MODEL_ID'].nunique()}")
print(f"Number of unique genes: {merged_gene_level["gene_symbol"].nunique()}")
print(
    f"Duplicate model-gene pairs: "
    f"{merged_gene_level.duplicated(subset=['SANGER_MODEL_ID', 'gene_symbol']).sum()}"
)

The merged dataset contained **948 unique cancer cell lines** and **597 unique mutated genes**, with **no duplicated cell line–gene pairs** identified. These checks confirmed that the merged dataset was suitable for construction of the binary mutation matrix used for subsequent machine learning analyses.

### 6.2.1. Construction of the Binary Feature Matrix

The objective of this project is to predict Trametinib sensitivity using biologically relevant features that would be available functional before screening.

The primary predictors therefore comprise binary gene mutation status, total mutation burden, MAPK pathway mutation burden and cancer type. Variables derived from the drug response experiment itself (e.g. AUC, RMSE and Z-score) were excluded because they would not be available when predicting drug sensitivity for an unseen sample and would therefore introduce data leakage. Similarly, detailed mutation annotation fields (e.g. genomic position and transcript identifiers) were excluded because the analysis was performed at the gene level.

The cleaned mutation dataset is currently stored in long format, where each row represents a single driver mutation identified within a cancer cell line. Since machine learning algorithms require one observation per row, the dataset was transformed into a wide-format feature matrix. Each cancer cell line therefore became a single observation, while each mutated gene became an individual binary predictor indicating the presence (1) or absence (0) of a mutation.

In [ ]:
# Create a binary mutation indicator
merged_gene_level["mutated"] = 1

# Convert from long to wide format
gene_matrix = (
    merged_gene_level
    .pivot_table(
        index="SANGER_MODEL_ID",
        columns="gene_symbol",
        values="mutated",
        aggfunc="max", # Included in case future datasets have duplicates
        fill_value=0 # if gene isnt present fill as 0
    )
    .astype(int)
)

### Validation of the Binary Gene Feature Matrix

In [ ]:
print(f"Feature matrix shape: {gene_matrix.shape}")

display(gene_matrix.head())

In [ ]:
print(f"Feature matrix shape: {gene_matrix.shape}")

print(f"Unique cell lines: {gene_matrix.index.nunique()}")

print(f"Unique genes: {gene_matrix.shape[1]}")

print(f"Missing values: {gene_matrix.isna().sum().sum()}")

The mutation data were successfully transformed into a binary feature matrix containing one row per cancer cell line and one column per mutated gene. The resulting matrix comprised 948 observations and 597 binary gene features with no missing values, confirming that the data were suitable for subsequent feature engineering and machine learning.

### Engineering the Mutation Burden Feature

In addition to individual gene mutations, the total number of mutated driver genes was calculated for each cancer cell line. This feature represents the overall driver mutation burden and was included because the cumulative number of driver mutations may contribute to differences in tumour biology and influence response to targeted therapies. Mutation burden was calculated by counting the number of unique mutated genes present within each cancer cell line.

In [ ]:
# Calculate the total number of mutated genes per cancer cell line

mutation_burden = (
    merged_gene_level
    .groupby("SANGER_MODEL_ID")
    .size()
    .rename("Mutation_Burden")
)

mutation_burden.head()

In [ ]:
print(f"Number of cell lines: {mutation_burden.shape[0]}")
print(f"Minimum mutation burden: {mutation_burden.min()}")
print(f"Maximum mutation burden: {mutation_burden.max()}")
print(f"Mean mutation burden: {mutation_burden.mean():.2f}")

mutation_burden.describe()

The mutation burden varied substantially across the 948 cancer cell lines, ranging from **1** to **114** mutated genes per cell line. The distribution was positively skewed, with a median of **5** mutations compared with a mean of **8.26**, indicating that most cell lines contained relatively few mutations while a smaller number exhibited much higher mutation burdens.

This variability supported inclusion of mutation burden as a continuous predictor, as differences in the overall number of mutated genes may contribute to variation in Trametinib sensitivity.

### Validation of the Mutation Burden Feature

The distribution of mutation burden was examined to confirm that the calculated values were biologically plausible and to identify any unexpected outliers before incorporating the feature into the machine learning dataset.

In [ ]:
# Plot distribution of mutation burden histogram
plt.figure(figsize=(8, 5))
plt.hist(mutation_burden, bins=30)
plt.xlabel("Mutation burden")
plt.ylabel("Number of cell lines")
plt.title("Distribution of mutation burden")
plt.show()

The distribution of mutation burden was strongly right-skewed, with the majority of cancer cell lines harbouring relatively few mutated genes and a small number of cell lines exhibiting substantially higher mutation burdens. Most cell lines contained fewer than 15 mutated genes, while only a limited number of highly mutated cell lines formed the long right-hand tail of the distribution.

This variability suggests that overall mutation burden differs considerably between cancer cell lines and may contribute to variation in Trametinib sensitivity. Mutation burden was therefore retained as a continuous predictor for subsequent machine learning analyses.

### Incorporation of Mutation Burden into the Feature Matrix

Following validation, mutation burden was incorporated into the feature matrix as an additional predictor alongside the binary gene mutation features.

In [ ]:
# Create a copy of the binary gene matrix
X_features = gene_matrix.copy()

# Add mutation burden
X_features["Mutation_Burden"] = mutation_burden

In [ ]:
print(f"Feature matrix shape: {X_features.shape}")

print(f"Missing mutation burden values: {X_features['Mutation_Burden'].isna().sum()}")

### Engineering and Validation of the MAPK Mutation Burden Feature

Because Trametinib inhibits MEK1/2 within the MAPK signalling pathway, the number of mutated MAPK-related genes was calculated for each cancer cell line. This feature provides a pathway-level summary of genomic alterations directly relevant to Trametinib mechanism of action and may capture biological information not represented by individual gene features alone.

In [ ]:
# Redefine the curated set of genes involved in MAPK signalling previously used in Notebook 1
# Mutations in these genes may influence sensitivity to the MEK inhibitor Trametinib
mapk_pathway_gene_set = {
    "KRAS", "NRAS", "HRAS",
    "NF1", "RASA1", "RASA2", "RASA3", "RASA4",
    "BRAF", "RAF1", "ARAF",
    "MAP2K1", "MAP2K2",
    "MAPK1", "MAPK3",
    "EGFR", "ERBB2", "ERBB3", "ERBB4",
    "FGFR1", "FGFR2", "FGFR3", "FGFR4",
    "MET", "KIT", "PDGFRA", "PDGFRB",
    "ALK", "RET", "ROS1",
    "SOS1", "SOS2", "GRB2", "SHC1",
    "DUSP4", "DUSP5", "DUSP6",
    "SPRY1", "SPRY2", "SPRY4",
    "MAP3K1", "MAP3K2", "MAP3K3", "MAP3K4",
    "MAP3K5", "MAP3K7", "MAP3K8",
    "MAP2K3", "MAP2K4", "MAP2K5", "MAP2K6", "MAP2K7",
    "MAPK7", "MAPK8", "MAPK9", "MAPK10",
    "MAPK11", "MAPK12", "MAPK13", "MAPK14"
}

# Count the number of mutated MAPK pathway genes in each cell line
mapk_mutation_burden = (
    merged_gene_level[
        # Retain only mutations occurring in MAPK pathway genes
        merged_gene_level["gene_symbol"].isin(mapk_pathway_gene_set)
    ]
    # Count MAPK mutations for each cell line
    .groupby("SANGER_MODEL_ID")
    .size()
    # Ensure every cell line is represented (assign zero where no MAPK mutations are present)
    .reindex(gene_matrix.index, fill_value=0)
    # Give the feature a descriptive name
    .rename("MAPK_Mutation_Burden")
)

In [ ]:
print(f"Number of cell lines: {mapk_mutation_burden.shape[0]}")
print(f"Missing values: {mapk_mutation_burden.isna().sum()}")

display(mapk_mutation_burden.describe())

The engineered MAPK Mutation Burden feature showed a relatively low mutation frequency across the dataset. Most cell lines contained no mutations within the selected MAPK pathway genes (median = 0), while a smaller proportion contained one or more alterations, with a maximum of six mutated pathway genes per cell line. This distribution indicates that although MAPK pathway mutations were uncommon, substantial variation existed between cell lines, supporting the inclusion of this biologically informed feature as a predictor of Trametinib sensitivity.

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    mapk_mutation_burden,
    bins=range(
        int(mapk_mutation_burden.max()) + 2
    ),
    align="left"
)

plt.xlabel("MAPK mutation burden")
plt.ylabel("Number of cell lines")
plt.title("Distribution of MAPK mutation burden")
plt.show()

The distribution of MAPK Mutation Burden was strongly right-skewed, with the majority of cell lines harbouring no detectable mutations within the curated MAPK gene set. A progressively smaller number of cell lines contained increasing numbers of pathway mutations, resulting in a long right-hand tail. This pattern is consistent with the expectation that only a subset of cancers exhibit multiple alterations affecting MAPK signalling.

Given that Trametinib directly targets the MAPK pathway through inhibition of MEK1/2, summarising the overall burden of pathway mutations provides a biologically meaningful feature that may capture pathway dysregulation more effectively than considering individual mutations alone.

### Incorporation of MAPK Mutation Burden into the Feature Matrix

Following validation, MAPK mutation burden was incorporated into the feature matrix as a pathway-level predictor relevant to Trametinib mechanism of action.

In [ ]:
# Add the engineered MAPK mutation burden feature to the predictor matrix
X_features["MAPK_Mutation_Burden"] = mapk_mutation_burden

In [ ]:
print(f"Feature matrix shape: {X_features.shape}")

print(
    f"Missing MAPK mutation burden values: "
    f"{X_features['MAPK_Mutation_Burden'].isna().sum()}"
)

The engineered MAPK Mutation Burden feature was successfully incorporated into the predictor matrix, increasing the total number of predictor variables to **599**. Validation confirmed that no missing values were introduced during feature engineering, ensuring that the complete feature matrix was suitable for downstream machine learning analyses without requiring additional imputation.

### Incorporation of Cancer Type

In addition to genomic features, cancer type was retained as a predictor because tissue lineage is known before treatment and may influence response to targeted therapies. This variable provides complementary biological information that is not captured by mutation status alone.

At this stage, cancer type was retained in its original categorical format rather than being one-hot encoded. One-hot encoding was instead performed within the machine learning preprocessing pipeline after the train-test split. This ensured that the encoder was fitted using only the training data during each cross-validation fold and subsequently applied to the validation and independent test datasets. Incorporating encoding within the pipeline provided a consistent preprocessing workflow for all machine learning models while preventing information from outside the training data from influencing feature construction.

In [ ]:
# Create a mapping between each cell line and its corresponding cancer type
cancer_type = (
    merged_gene_level
    .drop_duplicates(subset="SANGER_MODEL_ID")
    .set_index("SANGER_MODEL_ID")["CANCER_TYPE"]
)

In [ ]:
print(f"Number of cell lines: {cancer_type.shape[0]}")

print(f"Missing values: {cancer_type.isna().sum()}")

print(cancer_type.value_counts().head())

The cancer type variable was successfully extracted for all **948** cell lines, with **no missing values** identified. The dataset comprised **42 distinct cancer types**, with Non-Small Cell Lung Carcinoma representing the largest group (88 cell lines), followed by Small Cell Lung Carcinoma (57), Melanoma (54), Breast Carcinoma (50) and Colorectal Carcinoma (48). Retaining cancer type as a predictor enables the machine learning models to account for tissue-specific differences in Trametinib sensitivity that may not be explained by genomic mutations alone.

In [ ]:
# Add the cancer type feature to the predictor matrix
X_features["CANCER_TYPE"] = cancer_type

In [ ]:
print(X_features.shape)

The final feature matrix contained **948 cancer cell lines** and **600 predictor variables**, comprising binary gene mutation features, mutation burden, MAPK mutation burden and cancer type. This completed the feature engineering stage and produced the dataset used for subsequent machine learning model development.

### 6.2.2. Validation of the Final Feature Matrix

Following feature engineering, the complete feature matrix was validated to confirm the expected dimensions, ensure that no missing values had been introduced during feature integration, and verify that each observation continued to represent a unique cancer cell line. These checks ensure the dataset is suitable for machine learning prior to defining the target variable and performing the train-test split.

In [ ]:
print(f"Feature matrix shape: {X_features.shape}")

print(f"Unique cell lines: {X_features.index.nunique()}")

print(f"Missing values: {X_features.isna().sum().sum()}")

print(f"Duplicate indices: {X_features.index.duplicated().sum()}")

In [ ]:
X_features.info()

The completed feature matrix contained 948 observations and 600 predictor variables with no missing values or duplicate observations. The dataset comprised 597 binary gene mutation features, two engineered numerical features (mutation burden and MAPK mutation burden), and one categorical feature (cancer type). Following validation, the feature matrix was considered suitable for supervised machine learning.

# 7. Model Preparation

The engineered feature matrix was prepared for supervised machine learning to predict Trametinib sensitivity (LN_IC50). Prior to model training, the predictor variables (X) and target variable (y) were defined before the dataset was partitioned into independent training and test sets. Subsequent preprocessing and model optimisation were performed exclusively using the training data to prevent data leakage.

## 7.1 Definition of Predictor and Target Variables

The feature matrix was separated into predictor variables (X) and the target variable (y). The predictor variables comprised binary gene mutation features, mutation burden, MAPK mutation burden and cancer type, while the target variable was the continuous LN_IC50 value representing Trametinib sensitivity. Separating the predictors and target variable establishes a clear distinction between the input features and response variable prior to model development.

In [ ]:
# Create the target variable (one LN_IC50 value per cell line)
y = (
    merged_gene_level
    .drop_duplicates(subset="SANGER_MODEL_ID")
    .set_index("SANGER_MODEL_ID")["LN_IC50"]
)

# Predictor matrix
X = X_features.copy()

In [ ]:
print(f"Predictor matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

print(f"Predictor rows equal target rows: {len(X) == len(y)}")

print(f"Indices aligned before reordering: {X.index.equals(y.index)}")

# Reorder the target so that each LN_IC50 value matches the corresponding predictor row
y = y.reindex(X.index)

print(f"Indices aligned after reordering: {X.index.equals(y.index)}")

print(f"Missing target values: {y.isna().sum()}")

The predictor and target datasets contained the same 948 cancer cell lines but were initially stored in different row orders. The target variable was therefore reindexed to match the predictor matrix using Sanger Model IDs, ensuring that each feature vector was correctly paired with its corresponding LN_IC50 value before data splitting.

## 7.2. Train-Test Split

To provide an independent assessment of model performance, the dataset was divided into training and test sets using an 80:20 split. A fixed random state of 42 was used to ensure reproducibility.

The test set was reserved exclusively for final model evaluation and was not used during feature selection, preprocessing, model selection or hyperparameter optimisation. Model validation and hyperparameter tuning will instead be performed using 5-fold cross-validation within the training dataset.

In [ ]:
# Split the dataset into independent training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
print(f"Training predictors: {X_train.shape}")
print(f"Test predictors: {X_test.shape}")
print(f"Training target: {y_train.shape}")
print(f"Test target: {y_test.shape}")

print(f"\nTraining indices aligned: {X_train.index.equals(y_train.index)}")
print(f"Test indices aligned: {X_test.index.equals(y_test.index)}")

overlap = len(set(X_train.index).intersection(X_test.index))
print(f"Overlap between training and test cell lines: {overlap}")

The 80:20 split produced 758 training and 190 test observations, with predictor and target indices correctly aligned and no overlap between the two datasets. The test dataset was therefore reserved as an independent hold-out set and will remain untouched until final model evaluation. All subsequent preprocessing, feature selection, model selection and hyperparameter optimisation will be performed using the training data only.

## 7.3. Exploration of and Filtering by Gene Mutation Frequencies

Before applying feature selection, the frequency of each binary gene mutation feature was examined within the training dataset. This analysis was performed to understand how commonly individual gene mutations occurred and to inform the selection of an appropriate minimum frequency threshold for retaining gene features.

In [ ]:
# Count the number of training cell lines containing each mutated gene
gene_columns = X_train.columns[:597]
gene_counts = X_train[gene_columns].sum().sort_values()

In [ ]:
# Select only the binary gene mutation columns
gene_columns = X_train.columns[:597]

# Count the number of mutated cell lines for each gene
gene_counts = X_train[gene_columns].sum()

# Summary statistics
display(gene_counts.describe())

# Histogram
plt.figure(figsize=(10, 6))

plt.hist(
    gene_counts,
    bins=30,
    edgecolor="black"
)

plt.xlabel("Number of training cell lines with a mutation")
plt.ylabel("Number of genes")
plt.title("Distribution of gene mutation frequencies in the training set")

plt.show()

The frequency of gene mutations within the training dataset was highly skewed. Most genes were mutated in only a small number of cell lines, with a median mutation frequency of five and three-quarters of genes occurring in 11 or fewer cell lines. In contrast, a small number of genes were mutated very frequently, with the most common mutation observed in 510 training cell lines.

This distribution suggests that many genes provide limited information for model training because they occur only rarely. Consequently, a minimum mutation frequency threshold was applied using the training dataset to remove extremely rare mutation features while retaining recurrent genomic alterations that are more likely to contribute to prediction of Trametinib sensitivity.

In [ ]:
# Evaluate a range of minimum mutation frequency thresholds
thresholds = [1, 2, 5, 10, 20]

for t in thresholds:
    # Count genes that would be retained or removed at each threshold
    retained = (gene_counts >= t).sum()
    removed = (gene_counts < t).sum()

    print(
        f"Threshold ≥ {t:2}: "
        f"Retain {retained:3} genes | "
        f"Remove {removed:3} genes"
    )

The impact of several minimum mutation frequency thresholds was evaluated using the training dataset. Increasing the threshold progressively reduced the number of retained gene mutation features, removing increasingly rare mutations while simplifying the feature space.

A threshold of **five mutated cell lines** was selected as an appropriate balance between retaining biologically informative mutations and excluding extremely rare features that were unlikely to contribute meaningfully to model performance. This threshold retained **329** recurrent gene mutation features while removing **268** infrequently mutated genes, substantially reducing feature dimensionality without eliminating the majority of informative predictors.

In [ ]:
gene_counts.value_counts().sort_index().head(10)

The frequency distribution further demonstrated that many genes occurred in only a handful of cell lines, with 14 genes never mutated in the training dataset and a large proportion occurring in fewer than five cell lines. This reinforced the decision to remove very low-frequency mutation features prior to model development. The threshold was determined using only the training dataset and the resulting set of retained genes was applied unchanged to the independent test dataset, ensuring feature selection did not introduce data leakage.

### 7.3.1. Filtering Low-Frequency Gene Features

The selected minimum frequency threshold was applied to the training dataset to identify binary gene mutation features present in at least five training cell lines. The resulting feature set was then applied unchanged to both the training and test datasets.

In [ ]:
# Keep genes mutated in at least 5 training cell lines
retained_gene_columns = gene_counts[gene_counts >= 5].index

print(f"Genes retained: {len(retained_gene_columns)}")
print(f"Genes removed: {len(gene_columns) - len(retained_gene_columns)}")

### 7.3.2. Create the Filtered Dataset

In [ ]:
# Retain the selected gene mutation features together with the additional biological predictors 
X_train_filtered = X_train[
    retained_gene_columns.tolist() +
    [
        "Mutation_Burden",
        "MAPK_Mutation_Burden",
        "CANCER_TYPE"
    ]
].copy()

# Apply the identical feature selection to the independent test dataset
X_test_filtered = X_test[
    retained_gene_columns.tolist() +
    [
        "Mutation_Burden",
        "MAPK_Mutation_Burden",
        "CANCER_TYPE"
    ]
].copy()

In [ ]:
print(f"Training shape: {X_train_filtered.shape}")
print(f"Test shape: {X_test_filtered.shape}")

print(f"Training columns equal test columns: {list(X_train_filtered.columns) == list(X_test_filtered.columns)}")

print(f"Missing values (training): {X_train_filtered.isna().sum().sum()}")
print(f"Missing values (test): {X_test_filtered.isna().sum().sum()}")

The selected frequency threshold was successfully applied to the training dataset, reducing the binary gene feature set from 597 to 329 genes. The same retained gene set was subsequently applied to the independent test dataset, ensuring that both datasets contained an identical feature space while preventing information from the test set influencing feature selection. Following filtering, the predictor matrices contained 332 features, comprising 329 binary gene mutation features, mutation burden, MAPK mutation burden and cancer type, and were considered suitable for subsequent preprocessing and model development.

## 7.4. Feature Preprocessing

Before model training, the predictor data required preprocessing to ensure compatibility with the selected machine learning algorithms while preventing data leakage. As the feature matrix contained both binary, numerical and categorical variables, different preprocessing strategies were required for each feature type.

The binary gene mutation features were filtered using a predefined minimum frequency threshold determined from the training dataset. Mutation burden and MAPK mutation burden were retained as continuous numerical variables, while cancer type was converted to a numerical representation using one-hot encoding.

To ensure that preprocessing was performed independently within each cross-validation fold, all preprocessing steps were incorporated into scikit-learn pipelines rather than being applied globally before model training. The independent test dataset remained excluded from preprocessing, model selection and hyperparameter optimisation until final model evaluation.

### 7.4.1. Preprocessing Pipeline

Cancer type was one-hot encoded because it is a nominal categorical variable with no inherent ordering. Binary gene mutation features were retained without scaling, while the two continuous engineered features were standardised for models that are sensitive to feature magnitude. All preprocessing steps were incorporated into scikit-learn pipelines so that transformations were fitted using training data only.

In [ ]:
# Define the categorical predictor to be one-hot encoded
categorical_features = ["CANCER_TYPE"]

# Define the continuous numerical predictors
continuous_features = [
    "Mutation_Burden",
    "MAPK_Mutation_Burden"
]
# Identify the retained binary gene mutation features
gene_features = [
    column for column in X_train_filtered.columns
    if column not in categorical_features + continuous_features
]

# Verify the number of features in each category
print(f"Gene features: {len(gene_features)}")
print(f"Continuous features: {len(continuous_features)}")
print(f"Categorical features: {len(categorical_features)}")

The filtered predictor matrix comprised **329 binary gene mutation features**, **two continuous biological features** (Mutation Burden and MAPK Mutation Burden) and **one categorical feature** (Cancer Type). Separating the predictors according to their data type enabled appropriate preprocessing within the machine learning pipeline, with binary mutation features passed directly to the models, continuous variables retained as numerical features, and cancer type one-hot encoded during model training.

### 7.4.2. Preprocessing Pipeline for Scaled Models

The predictor variables required different preprocessing strategies according to their data type. The binary gene mutation features were retained without transformation, as they already represented binary indicators of mutation status. The continuous variables (Mutation Burden and MAPK Mutation Burden) were standardised using z-score scaling to place them on a common numerical scale, while the categorical cancer type variable was converted into binary indicator variables using one-hot encoding.

These preprocessing steps were combined within a scikit-learn `ColumnTransformer`, ensuring that each feature type received the appropriate transformation while maintaining a reproducible workflow. Integrating preprocessing within the machine learning pipeline ensured that all transformations were fitted using only the training data during cross-validation and then applied consistently to the validation and independent test datasets.

In [ ]:
# Apply feature-specific preprocessing using a ColumnTransformer
preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            # Retain binary gene mutation features without modification
            "genes",
            "passthrough",
            gene_features
        ),
        (
            # Standardise continuous biological features
            "continuous",
            StandardScaler(),
            continuous_features
        ),
        (
            # One-hot encode the categorical cancer type feature
            "cancer_type",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

### 7.4.3. Preprocessing Pipeline for Tree-Based Models

Unlike neural networks and other distance- or gradient-based algorithms, decision trees and tree ensemble methods are unaffected by the scale of the input features because predictions are based on feature splitting rather than numerical distance or gradient optimisation. Consequently, the continuous variables were retained in their original scale, while the categorical cancer type variable was one-hot encoded and the binary gene mutation features were passed through unchanged.

A second `ColumnTransformer` was therefore created specifically for tree-based models, allowing the same preprocessing workflow to be used without unnecessary feature scaling.

In [ ]:
# Create a preprocessing pipeline for tree-based models (no feature scaling required)
preprocessor_unscaled = ColumnTransformer(
    transformers=[
        (
            # Retain binary gene mutation features without modification
            "genes",
            "passthrough",
            gene_features
        ),
        (
            # Retain continuous biological features in their original scale
            "continuous",
            "passthrough",
            continuous_features
        ),
        (
            # One-hot encode the categorical cancer type feature
            "cancer_type",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

### 7.4.4. Validation of the Preprocessing Pipelines

The preprocessing pipelines were validated using the training dataset to confirm that categorical encoding, feature passthrough and numerical scaling were applied correctly before model training. Validation at this stage ensures that the transformed feature matrices have the expected dimensions and contain no missing values.

In [ ]:
# Fit the preprocessing pipeline for scaled models using the training dataset
X_train_scaled = preprocessor_scaled.fit_transform(X_train_filtered)
# Fit the preprocessing pipeline for tree-based models using the training dataset
X_train_unscaled = preprocessor_unscaled.fit_transform(X_train_filtered)

In [ ]:
print(f"Scaled training matrix shape: {X_train_scaled.shape}")
print(f"Unscaled training matrix shape: {X_train_unscaled.shape}")

In [ ]:
print(f"Missing values (scaled): {np.isnan(X_train_scaled).sum()}")
print(f"Missing values (unscaled): {np.isnan(X_train_unscaled).sum()}")

In [ ]:
encoder = preprocessor_scaled.named_transformers_["cancer_type"]

print(f"Number of cancer types: {len(encoder.categories_[0])}")

Validation confirmed that both preprocessing pipelines produced training matrices with **758 cell lines and 373 model-ready features**, with no missing values introduced during transformation. The reduction from the preprocessed feature set reflects retention of the **329 filtered gene mutation features**, two continuous variables and the expansion of cancer type into **42 one-hot encoded categories**.

Both the scaled and unscaled preprocessing pipelines produced identical feature dimensions, differing only in whether the continuous variables were standardised. This confirmed that the preprocessing strategy was functioning correctly before model development.

## 7.5. Model Evaluation Strategy

To ensure a fair comparison between algorithms, all machine learning models were evaluated using the same training, validation and testing strategy. Hyperparameter optimisation was performed using 5-fold cross-validation on the training dataset, while the independent test dataset remained completely unseen until final model evaluation.

The following performance metrics were reported for each model:

- Cross-validation R² (mean ± standard deviation)
- Independent Test R²
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)

Using both cross-validation and an independent test dataset enabled assessment of model generalisation during training while providing an unbiased estimate of predictive performance on previously unseen cell lines.

In [ ]:
# Create a reproducible 5-fold cross-validation strategy with shuffled observations
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

### 7.5.1. Model Evaluation Function

To ensure consistency across all machine learning models, a reusable evaluation function was implemented. This function performs cross-validation using the training dataset, fits the model to the complete training set, evaluates performance on the independent test set, and returns the evaluation metrics used throughout the project.

In [ ]:
def evaluate_model(
    model_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    cv
):
    """
    Evaluate a machine learning model using
    cross-validation and an independent test set.
    """

    # Cross-validation
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="r2"
    )

    # Fit the model using the complete training dataset
    pipeline.fit(X_train, y_train)

    # Generate predictions for the independent test dataset
    y_pred = pipeline.predict(X_test)

    # Calculate cross-validation and test performance metrics
    results = {
        "Model": model_name,
        "CV R² Mean": cv_scores.mean(),
        "CV R² SD": cv_scores.std(),
        "Test R²": r2_score(y_test, y_pred),
        "Test MAE": mean_absolute_error(y_test, y_pred),
        "Test RMSE": root_mean_squared_error(y_test, y_pred)
    }

    return results

### 7.5.2. Model Comparison Table

To facilitate comparison between machine learning algorithms, evaluation metrics from each model were stored in a single results table. This provides a consistent summary of model performance across cross-validation and the independent test dataset.

In [ ]:
# Create a list to store the evaluation results for each machine learning model
model_results = []

A second helper function was implemented to summarise the performance of all machine learning models. Following evaluation, the results were combined into a single table, rounded for presentation and automatically ranked according to independent test R². This provided a consistent framework for comparing predictive performance across all algorithms investigated.

In [ ]:
def update_results_table(model_results):
    # Convert the collected model results into a ranked comparison table
    return (
        pd.DataFrame(model_results)
        .round(3)   # Round performance metrics
        .sort_values("Test R²", ascending=False)    # Rank by independent test performance
        .reset_index(drop=True) # Reset row numbering
        .rename_axis("Rank")    # Label the index as 'Rank'
    )

# 8. Model Development

## 8.1. Baseline Model: Linear Regression

Linear Regression was selected as the baseline model because it provides a simple, interpretable benchmark against which the performance of more complex machine learning algorithms can be compared. Although drug response is unlikely to be explained entirely by linear relationships, establishing a baseline allows the value of increasingly sophisticated models to be objectively assessed.

In [ ]:
# Create a machine learning pipeline for Linear Regression
# The scaled preprocessor standardises continuous features, retains binary gene
# mutation features unchanged, and one-hot encodes cancer type
linear_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor_scaled),
        ("model", LinearRegression())
    ]
)

# Evaluate the Linear Regression model using 5-fold cross-validation
# and the independent test dataset
linear_results = evaluate_model(
    model_name="Linear Regression",
    pipeline=linear_pipeline,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

# Store the evaluation results for later comparison with other models
model_results.append(linear_results)

# Update the ranked model comparison table
results_df = update_results_table(model_results)

# Display the current model performance table
results_df

The baseline Linear Regression model showed poor predictive performance, with a mean cross-validation R² of -2.58 and a test R² of -0.60. Negative R² values indicate that the model performed worse than predicting the mean LN_IC50 value. This suggests that Trametinib response is not adequately captured by a simple additive linear relationship between the selected genomic and cancer-type features, providing a useful baseline against which more flexible nonlinear models can be compared.

## 8.2. Decision Tree Regression

Decision Tree Regression was selected as the first nonlinear model because it can capture threshold effects and complex interactions between genomic features without assuming a linear relationship between predictors and drug response. Unlike Linear Regression, decision trees partition the feature space into regions with similar outcomes, allowing more flexible modelling of biological relationships.

In [ ]:
# Create a machine learning pipeline for the Decision Tree model
# Tree-based models do not require feature scaling, so the unscaled
# preprocessor retains the continuous variables in their original scale
decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor_unscaled),
        (
            "model",
            DecisionTreeRegressor(
                random_state=42
            )
        )
    ]
)

# Evaluate the Decision Tree model using 5-fold cross-validation
# and the independent test dataset
decision_tree_results = evaluate_model(
    model_name="Decision Tree",
    pipeline=decision_tree_pipeline,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

# Store the evaluation results for later comparison with other models
model_results.append(decision_tree_results)

# Update the ranked model comparison table
results_df = update_results_table(model_results)

# Display the current model performance table
results_df

The baseline Decision Tree Regressor demonstrated improved performance compared with Linear Regression across all evaluation metrics. The mean cross-validation R² improved from -2.58 to -0.26, while the test R² increased from -0.60 to -0.42. Test MAE and RMSE also decreased, indicating more accurate predictions.

This improvement suggests that nonlinear relationships and interactions between genomic features contribute to Trametinib response. However, the model continued to show poor generalisation, indicating that the default Decision Tree configuration was unlikely to represent the optimal model complexity. Hyperparameter optimisation was therefore performed to improve predictive performance while reducing overfitting.

### 8.2.1. Decission Tree Hyperparameter Optimisation

Although the baseline Decision Tree model provides a simple and interpretable approach to prediction, its performance is highly dependent on the choice of hyperparameters. Hyperparameter optimisation was therefore performed using 5-fold cross-validation to identify a combination of parameters that improved predictive performance while reducing overfitting.

The optimisation investigated parameters controlling tree depth, the minimum number of samples required for node splitting and terminal leaf nodes, and the number of features considered when selecting the optimal split. The objective was to maximise cross-validation R² before evaluating the tuned model on the independent test dataset.

In [ ]:
# Define the hyperparameter search space for the Decision Tree model
decision_tree_param_grid = {
    "model__max_depth": [3, 5, 10, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": [None, "sqrt", "log2"]
}

The candidate hyperparameter combinations were evaluated using **GridSearchCV** with five-fold cross-validation. For each combination, the Decision Tree model was trained on four folds of the training dataset and validated on the remaining fold. This process was repeated until every fold had served as the validation set once, and the mean cross-validation R² was used to identify the optimal hyperparameter combination.

Only the training dataset was used during hyperparameter optimisation, ensuring that the independent test dataset remained unseen until final model evaluation.

In [ ]:
# Configure GridSearchCV to identify the optimal Decision Tree hyperparameters
grid_search = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=decision_tree_param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

The pipeline corresponding to the optimal hyperparameter combination identified by GridSearchCV is shown below. The figure illustrates the preprocessing workflow applied prior to model fitting and confirms that preprocessing and model training were performed within a single scikit-learn pipeline.

In [ ]:
# Perform the hyperparameter search using the training dataset
grid_search.fit(
    X_train_filtered,
    y_train
)

The optimal Decision Tree retained the filtered binary mutation features together with the continuous variables and one-hot encoded cancer type variables, resulting in a total of 373 predictor features. The best-performing hyperparameter combination is reported below.

In [ ]:
# Display the optimal hyperparameter combination
best_params_df = (
    pd.DataFrame(
        grid_search.best_params_,
        index=["Optimal Value"]
    )
    .T
)

best_params_df

In [ ]:
print(f"Best cross-validation R²: {grid_search.best_score_:.3f}")

### 8.2.2. Hyperparameter Optimisation Results

Hyperparameter optimisation substantially improved the Decision Tree model. The optimal model limited tree depth to five levels while requiring a minimum of five samples per terminal node. These constraints reduce model complexity and help prevent overfitting, leading to improved generalisation during cross-validation. The tuned model was subsequently evaluated on the independent test dataset to determine whether these improvements translated to previously unseen data.

In [ ]:
# Retrieve the Decision Tree model with the optimal hyperparameters
best_decision_tree = grid_search.best_estimator_

# Evaluate the tuned Decision Tree model using 5-fold cross-validation
# and the independent test dataset
tuned_decision_tree_results = evaluate_model(
    model_name="Decision Tree (Tuned)",
    pipeline=best_decision_tree,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

# Store the evaluation results for comparison with the other models
model_results.append(tuned_decision_tree_results)

# Update the ranked model comparison table
results_df = update_results_table(model_results)

# Display the updated model performance table
results_df

The tuned Decision Tree substantially outperformed both the baseline Decision Tree and the Linear Regression model. Hyperparameter optimisation increased the mean cross-validation R² from -0.264 to 0.234 and improved the independent test R² from -0.424 to 0.279. Test MAE and RMSE were also reduced by approximately 28–29%, indicating more accurate predictions on previously unseen cell lines.

The optimal model employed a relatively shallow tree (maximum depth = 5) with a minimum of five samples per terminal node, suggesting that restricting model complexity improved generalisation and reduced overfitting. These findings demonstrate the importance of hyperparameter optimisation when applying tree-based machine learning algorithms to high-dimensional genomic datasets.

## 8.3. Random Forest Regression - START FROM HERE

Random Forest Regression was selected as an ensemble-based extension of the Decision Tree model. By combining predictions from multiple decision trees trained on different bootstrap samples and subsets of features, Random Forest can reduce the variance and overfitting associated with a single decision tree while retaining the ability to model nonlinear relationships and interactions between genomic features.

Because tree-based models are not sensitive to feature scaling, the unscaled preprocessing pipeline was used.

In [ ]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor_unscaled),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
random_forest_results = evaluate_model(
    model_name="Random Forest",
    pipeline=random_forest_pipeline,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

model_results.append(random_forest_results)

results_df = pd.DataFrame(model_results).round(3)

results_df

The baseline Random Forest model achieved the strongest predictive performance observed so far. Compared with the tuned Decision Tree, the Random Forest further improved both cross-validation and independent test performance, increasing the mean cross-validation R² from 0.234 to 0.303 and the test R² from 0.279 to 0.303.

The close agreement between cross-validation and test performance suggests that the model generalises well to previously unseen cell lines. These findings are consistent with the expected behaviour of Random Forests, which reduce the variance associated with individual decision trees through bootstrap aggregation (bagging) while retaining the ability to model complex nonlinear relationships.

### 8.3.1. Random Forest Hyperparameter Optimisation

The baseline Random Forest achieved the strongest performance observed so far. Hyperparameter optimisation was therefore performed using 5-fold cross-validation to investigate whether model performance could be improved further. Parameters controlling the number of trees, tree depth, minimum samples required for splitting and terminal nodes, and the number of features considered at each split were evaluated.

In [ ]:
random_forest_param_grid = {
    "model__n_estimators": [100, 300, 500],
    "model__max_depth": [5, 10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": ["sqrt", "log2", None]
}

random_forest_grid_search = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=random_forest_param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

random_forest_grid_search.fit(
    X_train_filtered,
    y_train
)

In [ ]:
print("Best parameters:")
print(random_forest_grid_search.best_params_)

print(
    f"\nBest cross-validation R²: "
    f"{random_forest_grid_search.best_score_:.3f}"
)

In [ ]:
best_random_forest = random_forest_grid_search.best_estimator_

tuned_random_forest_results = evaluate_model(
    model_name="Random Forest (Tuned)",
    pipeline=best_random_forest,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

model_results.append(tuned_random_forest_results)

results_df = update_results_table(model_results)

results_df

### 8.3.2. Hyperparameter Optimisation Results

Hyperparameter optimisation produced a further improvement in Random Forest performance across both cross-validation and the independent test dataset. The tuned Random Forest achieved the highest predictive performance observed so far, increasing the mean cross-validation R² from 0.303 to 0.355 and the test R² from 0.303 to 0.328. Test MAE and RMSE also decreased, indicating more accurate prediction of Trametinib sensitivity.

These results suggest that careful optimisation of ensemble parameters further improves model generalisation and reinforces the suitability of tree-based ensemble methods for modelling complex genomic predictors of drug response.

In [ ]:
best_rf_params = pd.DataFrame(
    [random_forest_grid_search.best_params_]
)

best_rf_params

The optimal Random Forest consisted of 500 decision trees, with each tree considering the square root of the available features at each split. Individual trees were allowed to grow without a maximum depth constraint, while requiring a minimum of two samples in each terminal node. These hyperparameters produced the strongest predictive performance observed in this study, indicating that a large ensemble of diverse decision trees was able to capture complex nonlinear relationships between genomic alterations and Trametinib response while maintaining good generalisation to previously unseen cell lines.

## 8.4. Gradient Boosting Regression

Gradient Boosting Regression was selected as a sequential ensemble learning algorithm that builds decision trees iteratively, with each new tree attempting to correct the prediction errors made by the previous ensemble. Unlike Random Forest, which constructs trees independently using bootstrap aggregation, Gradient Boosting focuses on progressively reducing residual error and can often achieve superior predictive performance through careful optimisation of weak learners.

As with the previous tree-based models, feature scaling was not required and the unscaled preprocessing pipeline was used.

In [ ]:


gradient_boosting_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor_unscaled),
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        )
    ]
)

In [ ]:
gradient_boosting_results = evaluate_model(
    model_name="Gradient Boosting",
    pipeline=gradient_boosting_pipeline,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

model_results.append(gradient_boosting_results)

results_df = update_results_table(model_results)

results_df

The baseline Gradient Boosting model achieved the highest independent test performance observed so far, with a test R² of 0.363 and the lowest MAE and RMSE of all models evaluated. Although the tuned Random Forest achieved a marginally higher cross-validation R², Gradient Boosting demonstrated superior predictive performance on the independent test dataset.

This result suggests that sequential boosting of weak learners may better capture the complex nonlinear relationships underlying Trametinib response than bootstrap aggregation alone. Hyperparameter optimisation was subsequently performed to determine whether Gradient Boosting performance could be improved further.

### 8.4.1. Gradient Boosting Hyperparameter Optimisation

The baseline Gradient Boosting model achieved the strongest predictive performance observed so far. Hyperparameter optimisation was therefore performed using 5-fold cross-validation to investigate whether model performance could be improved further. Parameters controlling the number of boosting iterations, learning rate, tree depth, and the minimum number of samples required for node splitting and terminal leaf nodes were evaluated to identify the combination that maximised predictive performance while maintaining good generalisation to the independent test dataset.

In [ ]:
gradient_boosting_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [2, 3, 5],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

gradient_boosting_grid_search = GridSearchCV(
    estimator=gradient_boosting_pipeline,
    param_grid=gradient_boosting_param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

gradient_boosting_grid_search.fit(
    X_train_filtered,
    y_train
)

In [ ]:
best_gradient_boosting = gradient_boosting_grid_search.best_estimator_

tuned_gradient_boosting_results = evaluate_model(
    model_name="Gradient Boosting (Tuned)",
    pipeline=best_gradient_boosting,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

model_results.append(tuned_gradient_boosting_results)

results_df = update_results_table(model_results)

results_df

### 8.4.2. Tuned Model Performance

Hyperparameter optimisation produced a modest but consistent improvement in Gradient Boosting performance. The tuned model achieved the highest predictive performance of all models evaluated, increasing the independent test R² from 0.363 to 0.376 while also reducing both MAE and RMSE.

The tuned Gradient Boosting model therefore represents the best-performing algorithm evaluated in this study and demonstrates that sequential boosting of weak learners is well suited to modelling the complex nonlinear relationships between genomic alterations and Trametinib response.

## 8.5. XGBoost

Given the strong performance of Gradient Boosting, XGBoost was evaluated as an advanced gradient boosting implementation incorporating regularisation, shrinkage and efficient tree construction.

In [ ]:
xgboost_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor_unscaled),
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
xgboost_results = evaluate_model(
    model_name="XGBoost",
    pipeline=xgboost_pipeline,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

model_results.append(xgboost_results)

results_df = update_results_table(model_results)

results_df

The baseline XGBoost model was evaluated to determine whether an advanced implementation of gradient boosting could improve predictive performance compared with the previously evaluated ensemble methods. As XGBoost contains several important hyperparameters controlling model complexity and regularisation, optimisation was subsequently performed using GridSearchCV.

### 8.5.1. Hyperparameter Optimisation

Hyperparameter optimisation was performed using 5-fold cross-validation. The search investigated the influence of tree depth, learning rate, number of boosting iterations, row subsampling, column subsampling and minimum child weight on predictive performance.

In [ ]:
xgboost_param_grid = {
    "model__n_estimators": [300, 500],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 5],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__min_child_weight": [1, 3]
}

In [ ]:
xgboost_grid_search = GridSearchCV(
    estimator=xgboost_pipeline,
    param_grid=xgboost_param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

xgboost_grid_search.fit(
    X_train_filtered,
    y_train
)

In [ ]:
print("Best parameters:")
print(xgboost_grid_search.best_params_)

print(
    f"\nBest cross-validation R²: "
    f"{xgboost_grid_search.best_score_:.3f}"
)

In [ ]:
best_xgboost_params = pd.DataFrame(
    [xgboost_grid_search.best_params_]
)

best_xgboost_params

### 8.5.2. Interpretation of Optimal Hyperparameters

The optimal XGBoost model used relatively shallow trees (maximum depth = 3) together with a reduced learning rate of 0.05 and 300 boosting iterations. Both row and feature subsampling were set to 0.8, introducing additional regularisation by training individual trees on subsets of the available observations and predictors. A minimum child weight of 3 further constrained tree growth.

Together, these settings indicate that the best-performing XGBoost configuration favoured a moderately regularised model rather than highly complex individual trees. The tuned model achieved a cross-validation R² of 0.352, indicating substantially improved generalisation compared with the baseline XGBoost model.

In [ ]:
best_xgboost = xgboost_grid_search.best_estimator_

tuned_xgboost_results = evaluate_model(
    model_name="XGBoost (Tuned)",
    pipeline=best_xgboost,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

model_results.append(tuned_xgboost_results)

results_df = update_results_table(model_results)

results_df

### 8.5.3. Comparison of Gradient Boosting and XGBoost

The tuned XGBoost model achieved performance comparable to the tuned Gradient Boosting model, producing a cross-validation R² of 0.352 and an independent test R² of 0.367. Although XGBoost achieved the lowest mean absolute error (MAE = 1.270), the tuned Gradient Boosting model retained the highest test R² (0.376) and the lowest RMSE (1.760), indicating slightly superior overall predictive performance.

These findings suggest that both boosting algorithms effectively captured nonlinear relationships within the genomic feature set. However, the additional regularisation and optimisation strategies implemented within XGBoost did not translate into a clear improvement over the classical Gradient Boosting implementation for this dataset.

## 8.6. Neural Network Regression
 Neural Networks were evaluated as a flexible nonlinear modelling approach capable of learning complex relationships between genomic features and Trametinib response. Unlike tree-based methods, neural networks learn hierarchical feature representations through multiple interconnected layers of neurons, allowing complex nonlinear interactions to be modelled directly from the data.

As neural networks are sensitive to the scale of input features, the scaled preprocessing pipeline was used.

In [ ]:
neural_network_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor_scaled),
        (
            "model",
            MLPRegressor(
                hidden_layer_sizes=(100,),
                activation="relu",
                solver="adam",
                alpha=0.0001,
                learning_rate_init=0.001,
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [ ]:
neural_network_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor_scaled),
        (
            "model",
            MLPRegressor(
                hidden_layer_sizes=(100,),
                activation="relu",
                solver="adam",
                alpha=0.0001,
                learning_rate_init=0.001,
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [ ]:
neural_network_results = evaluate_model(
    model_name="Neural Network",
    pipeline=neural_network_pipeline,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

model_results.append(neural_network_results)

results_df = update_results_table(model_results)

results_df

### 8.6.1. Hyperparameter Optimisation

Hyperparameter optimisation investigated the influence of network architecture, regularisation strength and learning rate on predictive performance. Optimisation was performed using 5-fold cross-validation.

In [ ]:
neural_network_param_grid = {
    "model__hidden_layer_sizes": [
        (50,),
        (100,),
        (100, 50)
    ],
    "model__alpha": [
        0.0001,
        0.001,
        0.01
    ],
    "model__learning_rate_init": [
        0.001,
        0.01
    ]
}

In [ ]:
neural_network_grid_search = GridSearchCV(
    estimator=neural_network_pipeline,
    param_grid=neural_network_param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

neural_network_grid_search.fit(
    X_train_filtered,
    y_train
)

In [ ]:
print("Best parameters:")
print(neural_network_grid_search.best_params_)

print(
    f"\nBest cross-validation R²: "
    f"{neural_network_grid_search.best_score_:.3f}"
)

In [ ]:
best_neural_network_params = pd.DataFrame(
    [neural_network_grid_search.best_params_]
)

best_neural_network_params

In [ ]:
best_neural_network = neural_network_grid_search.best_estimator_

tuned_neural_network_results = evaluate_model(
    model_name="Neural Network (Tuned)",
    pipeline=best_neural_network,
    X_train=X_train_filtered,
    X_test=X_test_filtered,
    y_train=y_train,
    y_test=y_test,
    cv=cv
)

model_results.append(tuned_neural_network_results)

results_df = update_results_table(model_results)

results_df

# 9. Model Interpretation

The tuned XGBoost model was selected for interpretation because it achieved predictive performance comparable to the best-performing model while providing robust methods for assessing feature importance. Understanding which genomic features contribute most strongly to Trametinib sensitivity provides biological insight beyond overall prediction accuracy.

In [ ]:
xgb_model = best_xgboost.named_steps["model"]
preprocessor = best_xgboost.named_steps["preprocessing"]

In [ ]:
feature_names = preprocessor.get_feature_names_out()

len(feature_names)

## 9.1. Feature Importance

Feature importance scores were extracted from the tuned XGBoost model to identify the genomic features contributing most strongly to prediction of Trametinib sensitivity.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(20)

In [ ]:
# Copy the top 20 features
top20 = feature_importance.head(20).copy()

# Clean feature names
top20["Feature"] = (
    top20["Feature"]
    .str.replace("genes__", "", regex=False)
    .str.replace("continuous__", "", regex=False)
    .str.replace("cancer_type__CANCER_TYPE_", "", regex=False)
    .str.replace("_", " ", regex=False)
)

# Plot smallest at the bottom, largest at the top
top20 = top20.sort_values("Importance")

# Create colour gradient
norm = mpl.colors.Normalize(
    vmin=top20["Importance"].min(),
    vmax=top20["Importance"].max()
)

colours = plt.cm.viridis(norm(top20["Importance"]))

# Create figure
plt.figure(figsize=(11, 8))

plt.barh(
    top20["Feature"],
    top20["Importance"],
    color=colours,
    edgecolor="black",
    linewidth=0.5
)

# Labels and title
plt.xlabel("Feature Importance", fontsize=16)
plt.ylabel("")

plt.title(
    "Top Predictors of Trametinib Sensitivity",
    fontsize=20,
    fontweight="bold",
    pad=15
)

# Increase tick label sizes
plt.xticks(fontsize=13)
plt.yticks(fontsize=15)


# Get current axis
ax = plt.gca()

# Add subtle grid
ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

# Clean up axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
best_gradient_boosting = gradient_boosting_grid_search.best_estimator_

In [ ]:
y_pred_gb = best_gradient_boosting.predict(X_test_filtered)

In [ ]:
plt.figure(figsize=(8, 8))

plt.scatter(
    y_test,
    y_pred_gb,
    s=70,
    alpha=0.7,
    edgecolor="black",
    linewidth=0.3
)

# Perfect prediction line
lims = [
    min(y_test.min(), y_pred_gb.min()),
    max(y_test.max(), y_pred_gb.max())
]

plt.plot(
    lims,
    lims,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Perfect Prediction"
)

plt.xlim(lims)
plt.ylim(lims)

# Larger fonts
plt.xlabel("Observed LN(IC50)", fontsize=16)
plt.ylabel("Predicted LN(IC50)", fontsize=16)

plt.title(
    "Observed vs Predicted Trametinib Response\n(Tuned Gradient Boosting)",
    fontsize=18,
    fontweight="bold",
    pad=15
)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)


plt.legend(
    fontsize=13,
    frameon=False,
    loc="upper left"
)

# Remove unnecessary borders
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.text(
    0.05,
    0.90,
    "Test R² = 0.376\nMAE = 1.284\nRMSE = 1.760",
    transform=ax.transAxes,
    fontsize=13,
    verticalalignment="top",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="grey")
)

plt.show()




### 9.2. Observed vs Predicted Trametinib Response

The tuned Gradient Boosting model demonstrated a clear positive relationship between observed and predicted LN(IC50) values on the independent test dataset, indicating that the model successfully captured meaningful variation in Trametinib sensitivity.

Predictions were generally most accurate for cell lines with intermediate levels of drug sensitivity. In contrast, the model tended to underestimate the most sensitive cell lines (low LN(IC50) values), with predictions shifted towards the centre of the response distribution. This regression-to-the-mean behaviour is common in machine learning regression models trained using squared error loss and suggests that additional biological features, such as gene expression or copy number alterations, may be required to improve prediction of extreme drug responses.

## 9.3. Overal Model Comparison



The performance of all baseline and tuned models was compared using 5-fold cross-validation and the independent test dataset. Models were assessed using R², mean absolute error (MAE) and root mean squared error (RMSE).

Tree-based ensemble methods consistently outperformed Linear Regression, the individual Decision Tree and the Neural Network. Hyperparameter optimisation improved performance across all model families where tuning was performed.

The tuned Gradient Boosting model achieved the highest test R² (0.376) and lowest RMSE (1.760), while tuned XGBoost achieved the lowest MAE (1.270) and comparable test R² (0.367). These results indicate that boosting-based ensemble methods provided the strongest overall predictive performance for Trametinib sensitivity using the available genomic and cancer-type features.


In [ ]:
results_df

In [ ]:
# Sort models by test R² so the best model appears at the top
plot_results = (
    results_df
    .reset_index()
    .sort_values("Test R²", ascending=True)
)

# Highlight the top two models
best_model = "Gradient Boosting (Tuned)"
second_model = "XGBoost (Tuned)"

bar_colours = []

for model in plot_results["Model"]:
    if model == best_model:
        bar_colours.append("forestgreen")
    elif model == second_model:
        bar_colours.append("mediumseagreen")
    else:
        bar_colours.append("lightgrey")

# Create figure
fig, ax = plt.subplots(figsize=(11, 8))

bars = ax.barh(
    plot_results["Model"],
    plot_results["Test R²"],
    color=bar_colours,
    edgecolor="black",
    linewidth=0.5
)

# Add zero reference line
ax.axvline(
    x=0,
    color="black",
    linestyle="--",
    linewidth=1
)

# Add R² value labels to each bar
for bar, value in zip(bars, plot_results["Test R²"]):
    y_position = bar.get_y() + bar.get_height() / 2

    if value >= 0:
        ax.text(
            value + 0.015,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=12
        )
    else:
        ax.text(
            value - 0.015,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="right",
            fontsize=12
        )

# Labels and title
ax.set_xlabel(
    r"Independent Test $R^2$",
    fontsize=16
)

ax.set_ylabel("")

ax.set_title(
    "Comparison of Machine Learning Models for Predicting Trametinib Response",
    fontsize=19,
    fontweight="bold",
    pad=16
)

# Larger tick labels
ax.tick_params(
    axis="x",
    labelsize=13
)

ax.tick_params(
    axis="y",
    labelsize=13
)

# Light grid to aid interpretation
ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

# Give enough space for labels
ax.set_xlim(
    min(plot_results["Test R²"]) - 0.12,
    max(plot_results["Test R²"]) + 0.12
)
# Remove unnecessary borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


plt.tight_layout()
plt.show()

The comparison demonstrates a clear improvement in predictive performance as increasingly flexible nonlinear and ensemble approaches were introduced. Linear Regression and the untuned Decision Tree produced negative test R² values, whereas tuned tree-based ensemble models achieved consistently positive performance. Gradient Boosting and XGBoost produced the strongest results, while the Neural Network remained comparatively weak even after hyperparameter optimisation.

These findings suggest that the structured, sparse genomic feature set was particularly well suited to boosted decision-tree methods.

# 10. Discussion

This study investigated the ability of machine learning models to predict Trametinib sensitivity across cancer cell lines using genomic mutation features, cancer type and a biologically engineered MAPK mutation burden feature. Multiple regression algorithms were evaluated using identical training, validation and independent test datasets to identify the modelling approach that provided the strongest predictive performance.

## Comparison of Machine Learning Models

Model performance improved progressively as increasingly flexible nonlinear algorithms were introduced. Linear Regression performed poorly (Test R² = -0.603), indicating that simple linear relationships were insufficient to explain variation in Trametinib response. A single Decision Tree also demonstrated limited predictive performance, although hyperparameter optimisation substantially improved its ability to generalise.

Ensemble tree methods consistently produced the strongest predictive performance. Random Forest improved upon the individual Decision Tree by reducing variance through bootstrap aggregation, while Gradient Boosting and XGBoost achieved the highest predictive accuracy by sequentially correcting residual prediction errors. The tuned Gradient Boosting model achieved the highest independent test performance (Test R² = 0.376), while the tuned XGBoost model produced comparable predictive accuracy (Test R² = 0.367) and the lowest mean absolute error. These findings suggest that boosting-based ensemble methods were particularly well suited to modelling the nonlinear relationships present within the genomic predictor set.

Despite their flexibility, Artificial Neural Networks performed substantially worse than the tree-based ensemble methods. Even after hyperparameter optimisation, the tuned neural network achieved only a Test R² of 0.055. This likely reflects the relatively modest sample size (758 training cell lines) and the structured, sparse nature of the mutation data, for which gradient-boosted decision trees are generally more effective than deep learning approaches.

## Biological Interpretation

Feature importance analysis demonstrated that the model learned biologically meaningful relationships rather than arbitrary statistical associations. The strongest predictor was BRAF mutation status, consistent with the mechanism of action of Trametinib as a MEK1/2 inhibitor acting within the MAPK signalling pathway. Additional high-ranking features included KRAS, NRAS and ERBB2, all of which are established regulators of MAPK signalling and therefore plausible determinants of drug response.

Cancer type also contributed substantially to prediction, with Melanoma, Colorectal Carcinoma and Small Cell Lung Carcinoma among the most influential features. This reflects the well-established observation that the biological consequences of MAPK pathway activation depend on tumour lineage and cellular context.

Importantly, the engineered MAPK Mutation Burden feature ranked among the most informative predictors. This demonstrates that biologically informed feature engineering can improve model performance by summarising multiple functionally related genomic alterations into a single predictor representing pathway dysregulation rather than individual mutations alone.

# 11. Limitations

Although the tuned Gradient Boosting model achieved the strongest predictive performance, approximately 62% of the variation in Trametinib response remained unexplained. This highlights the complexity of drug response biology and indicates that mutation data alone provide an incomplete description of therapeutic sensitivity.

Several additional molecular data types were unavailable for this study. Gene expression, copy number alterations, DNA methylation, protein abundance and phosphoproteomic measurements all contribute to MAPK pathway activity and may substantially improve predictive performance if incorporated into future models. Similarly, tumour microenvironment interactions, which are absent from cancer cell line models, represent an important determinant of clinical drug response.

Feature selection was intentionally conservative to minimise overfitting, removing genes mutated in fewer than five training cell lines. While this improved model robustness, rare driver mutations with genuine biological importance may also have been excluded.

Finally, model performance was evaluated using cancer cell lines rather than patient samples. Although cell line models provide valuable experimental systems, further validation using independent patient-derived datasets would be required before translation to clinical prediction.

# 12. Future Work

Several opportunities exist to further improve predictive performance. Incorporating multi-omics datasets, including gene expression, copy number variation and proteomic data, would provide a more comprehensive representation of tumour biology than mutation data alone.

Additional biologically informed feature engineering could also improve prediction by summarising pathway activity rather than considering individual genomic alterations independently. Explainable artificial intelligence approaches, such as SHAP analysis, could further increase model interpretability by quantifying feature contributions for individual predictions.

Finally, extending the modelling framework to additional targeted therapies would enable investigation of whether pathway-informed machine learning approaches generalise across different classes of precision oncology drugs.

# 13.  Conclusions

This study demonstrates that machine learning models can predict variation in Trametinib sensitivity using genomic mutation features and cancer type, with tree-based ensemble methods providing the strongest predictive performance. Among the evaluated algorithms, the tuned Gradient Boosting model achieved the highest independent test performance (Test R² = 0.376), closely followed by tuned XGBoost.

Feature importance analysis demonstrated that the models identified biologically plausible predictors, including BRAF, KRAS, NRAS and the engineered MAPK Mutation Burden feature, providing confidence that the algorithms learned meaningful biological relationships rather than spurious statistical associations.

Overall, these findings demonstrate the value of combining biologically informed feature engineering with modern machine learning approaches to model drug response in cancer. Although predictive performance remains limited by the available data, this work provides a framework that could be extended through integration of additional molecular datasets and external clinical validation.

# 14. Project Summary

This project developed and evaluated machine learning models to predict Trametinib sensitivity across 948 cancer cell lines using genomic mutation data, cancer type and a biologically informed MAPK mutation burden feature. Multiple regression algorithms were systematically compared using identical training, validation and independent test datasets.

### Key Findings

- **The tuned Gradient Boosting model achieved the strongest predictive performance**, with an independent test R² of **0.376**, followed closely by the tuned XGBoost model (Test R² = **0.367**).

- **Tree-based ensemble methods consistently outperformed Linear Regression, individual Decision Trees and Artificial Neural Networks**, demonstrating that nonlinear boosting approaches were best suited to modelling structured genomic mutation data.

- **Feature importance analysis identified biologically meaningful predictors**, with **BRAF mutation status**, **MAPK Mutation Burden**, **KRAS**, **NRAS** and **cancer type** ranking among the most influential features. These findings are consistent with the established mechanism of action of Trametinib and provide confidence that the models learned biologically relevant relationships.

- **Biologically informed feature engineering proved valuable**, with the engineered MAPK Mutation Burden feature emerging as one of the strongest predictors of drug response, demonstrating the benefit of incorporating pathway-level biological knowledge into machine learning workflows.

### Overall Conclusion

This study demonstrates that combining biologically informed feature engineering with modern machine learning methods can produce meaningful predictions of targeted therapy response. While predictive performance remains limited by the availability of genomic mutation data alone, the modelling framework developed here provides a robust foundation that could be extended through integration of additional multi-omics datasets and validation in independent patient cohorts.